In [6]:
# file: scripts/preproc_clustering_like_deconv.py
# Make clustering features share the SAME preprocess as your deconvolution:
# CP10k -> joint per-gene winsorize -> asinh; markers-first, then HVG, and FORCE-IN markers to HVG.

import os, warnings
from typing import Dict, List
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# ============== Config ==============
TRAIN_AD_PATH = "../train_data/train_adata.h5ad"
TEST_AD_PATH  = "../test_data/test_adata.h5ad"

LABEL_COL     = "highLevelType"
SAMPLE_COL    = "Sample"
CT_COL        = "highLevelType"
BATCH_KEY_ALL = "set"              # 'train' / 'test'

TARGET_ORDER   = ["T","B","Endothelial","Fibroblast","Plasmablast","Myofibroblast","NK","Myeloid","Mast"]
MARKER_TOPK_CT = 120
N_TOP_HVG      = 2000              # baseline HVG size (may expand after forcing markers)
N_PCS_H          = 50
N_PCS_M          = 50

STANDARDIZE_BEFORE_COMBINE = True

# match your deconv preprocess
GLOBAL_WINSORIZE = True
GLOBAL_WINSOR_Q  = 0.995
GLOBAL_ASINH     = True
GLOBAL_ASINH_C   = 1.0

OUT_DIR = "./outputs_features_csv/new_5/"
os.makedirs(OUT_DIR, exist_ok=True)

# ============== Utils ==============
def _to_dense32(X):
    """
    Robust 2D dense float32 (C-order):
    - 支援 scipy.sparse (用 .A / .toarray())
    - NumPy 2.0：允許必要時複製，之後再強制 C-contiguous
    """
    # 1) 取 dense
    X = X.A if hasattr(X, "A") else (X.toarray() if hasattr(X, "toarray") else X)
    # 2) 允許必要時複製來轉 dtype（避免 np.array(..., copy=False) 的報錯）
    X = np.asarray(X, dtype=np.float32)
    # 3) 保證 C-order，避免之後串接/quantile 的奇怪坑
    if not X.flags.c_contiguous:
        X = np.ascontiguousarray(X)
    # 4) 形狀檢查
    if X.ndim != 2:
        raise ValueError(f"期望 2D 矩陣，拿到 shape={X.shape}")
    return X



def joint_cp10k_winsor_asinh(
    ad_tr: sc.AnnData,
    ad_te: sc.AnnData,
    do_winsor: bool = True,
    q: float = 0.995,
    do_asinh: bool = True,
    c: float = 1.0
) -> tuple[sc.AnnData, sc.AnnData]:
    """
    CP10k（不 log1p）→ 在 train+test 聯合空間對每個「基因」做 winsor（上分位裁切）→ 可選 asinh。
    保證：
      - train/test 用同一組且同順序的基因
      - X 一律為 dense np.ndarray（float32，C-order）
      - per-gene 的上分位向量 up 與 (n_genes,) 對齊；broadcast 安全
      - 若 numpy 路徑仍出錯，退回到 pandas 的 quantile 計算
    """
    # 1) 對齊 gene 交集與順序
    genes_tr = ad_tr.var_names.astype(str)
    genes_te = ad_te.var_names.astype(str)
    genes = genes_tr.intersection(genes_te)
    if len(genes) == 0:
        raise ValueError("Train/Test 沒有共同基因。")
    ad_tr = ad_tr[:, genes].copy()
    ad_te = ad_te[:, genes].copy()

    # 2) CP10k（不 log1p；後面會 asinh）
    sc.pp.normalize_total(ad_tr, target_sum=1e4)
    sc.pp.normalize_total(ad_te, target_sum=1e4)

    # 3) 強制 dense + float32 + C-order

    Xtr = _to_dense32(ad_tr.X)
    Xte = _to_dense32(ad_te.X)

    # 4) 形狀護欄
    if Xtr.shape[1] != Xte.shape[1]:
        raise ValueError(f"train/test 基因數不一致：{Xtr.shape[1]} vs {Xte.shape[1]}")
    n_genes = Xtr.shape[1]

    # 5) 聯合 winsor（per-gene）
    if do_winsor:
        if not (0.5 < q < 1.0):
            raise ValueError(f"winsor 分位數 q 應在 (0.5,1.0)，目前 q={q}")
        try:
            # 最常見且最快的路徑
            joint = np.concatenate([Xtr, Xte], axis=0)             # (n_cells_all × n_genes)，保證是 ndarray
            up = np.quantile(joint, q, axis=0).astype(np.float32)  # (n_genes,)
        except Exception as e:
            # 退避：改用 pandas 計算 per-gene 分位
            # 避免任何 __array_function__ 的坑（例如隱性 sparse）
            df_tr = pd.DataFrame(Xtr)
            df_te = pd.DataFrame(Xte)
            up = pd.concat([df_tr, df_te], axis=0).quantile(q, axis=0).astype(np.float32).values
        if up.shape != (n_genes,):
            raise ValueError(f"內部錯誤：up 形狀應為 ({n_genes},)，拿到 {up.shape}")

        # 明確 broadcast，避免 numpy 的隱式規則出意外
        np.minimum(Xtr, np.broadcast_to(up, Xtr.shape), out=Xtr)
        np.minimum(Xte, np.broadcast_to(up, Xte.shape), out=Xte)
        np.clip(Xtr, 0.0, None, out=Xtr)
        np.clip(Xte, 0.0, None, out=Xte)

    # 6) asinh 變換（可選）
    if do_asinh:
        invc = 1.0 / float(c)
        Xtr = np.arcsinh(Xtr * invc, dtype=np.float32)
        Xte = np.arcsinh(Xte * invc, dtype=np.float32)

    # 7) 回寫
    ad_tr.X = Xtr
    ad_te.X = Xte
    return ad_tr, ad_te


def pseudobulk_by_sample_ct(ad: sc.AnnData, ct_col: str, sample_col: str) -> pd.DataFrame:
    # 保證 meta 欄位是字串，避免 groupby 出怪
    obs = ad.obs[[ct_col, sample_col]].astype(str).copy()

    cols, names = [], []
    # 用 pandas 的 groupby(indices) 取得每個 (sample, ct) 的 cell 索引
    for (s, ct), idx in obs.groupby([sample_col, ct_col]).indices.items():
        if len(idx) == 0:
            continue
        # 取出子矩陣（保持 2D）
        Xsub = ad.X[idx, :]
        # to dense
        if hasattr(Xsub, "toarray"):
            Xsub = Xsub.toarray()
        elif hasattr(Xsub, "A"):
            Xsub = Xsub.A
        else:
            Xsub = np.asarray(Xsub)
        # 強制 float32、C-order、2D 形狀 (n_cells × n_genes)
        Xsub = np.asarray(Xsub, dtype=np.float32, order="C")
        if Xsub.ndim == 0:
            # 單細胞且單基因時可能成 scalar；改成 (1×1)
            Xsub = Xsub.reshape(1, 1)
        elif Xsub.ndim == 1:
            # 只有一個細胞或只剩一個基因 → 變成 (1 × n_genes)
            Xsub = Xsub.reshape(1, -1)

        # 平均成 pseudobulk（1 × n_genes），塞到 cols
        cols.append(Xsub.mean(axis=0).ravel())
        names.append(f"{s}|{ct}")

    if not cols:
        raise ValueError("pseudobulk_by_sample_ct 產生空結果：檢查 Sample 與 CT 欄位是否正確，或是否沒有交集細胞。")

    return pd.DataFrame(
        np.column_stack(cols),
        index=ad.var_names.astype(str),
        columns=names
    )

def combat_on_pb(pb: pd.DataFrame) -> pd.DataFrame:
    ad = sc.AnnData(pb.T.copy())
    ad.obs["batch"] = pd.Categorical([c.split("|",1)[0] for c in pb.columns])  # batch = Sample
    sc.pp.combat(ad, key="batch")
    return pd.DataFrame(ad.X.T, index=pb.index, columns=pb.columns)

def stratified_marker_by_ct(pb: pd.DataFrame, topk_per_ct: int, ct_order: List[str]) -> Dict[str, List[str]]:
    """Per-CT log2FC vs rest（pseudobulk, ComBat 後）；回傳每 CT 的 topK gene list。"""
    cts = sorted({c.split("|",1)[1] for c in pb.columns})
    order = [ct for ct in ct_order if ct in cts] + [ct for ct in cts if ct not in ct_order]
    mean_all = pb.mean(axis=1) + 1e-9
    out: Dict[str, List[str]] = {}
    denom = max(1, len(cts)-1)
    for ct in order:
        cols = [c for c in pb.columns if c.endswith("|"+ct)]
        if not cols: out[ct] = []; continue
        m_ct = pb[cols].mean(axis=1)
        rest = (mean_all * len(cts) - m_ct) / denom
        sc_ct = np.log2((m_ct+1e-9)/(rest+1e-9)).sort_values(ascending=False)
        out[ct] = list(sc_ct.index[:topk_per_ct])
    return out

def _safe_union_markers(m_by_ct: Dict[str, List[str]], var_names) -> List[str]:
    s = set()
    for gl in m_by_ct.values(): s.update(gl)
    return sorted([g for g in s if g in set(var_names)])

def _stdz(X: np.ndarray) -> np.ndarray:
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True) + 1e-9
    return (X - mu) / sd

def _harmony_on_subset(adata: sc.AnnData, genes: List[str], batch_key: str, n_pcs: int) -> np.ndarray:
    ad = adata[:, genes].copy()
    sc.pp.scale(ad, max_value=10)
    sc.tl.pca(ad, n_comps=min(n_pcs, ad.n_vars, ad.n_obs), svd_solver="arpack")
    sc.external.pp.harmony_integrate(ad, key=batch_key)
    if "X_pca_harmony" not in ad.obsm:
        raise RuntimeError("Harmony on subset failed.")
    return ad.obsm["X_pca_harmony"]

# ============== Pipeline ==============
# Load
ad_tr = sc.read_h5ad(TRAIN_AD_PATH)
ad_te = sc.read_h5ad(TEST_AD_PATH)
for need in (LABEL_COL, SAMPLE_COL, CT_COL):
    if need not in ad_tr.obs: raise KeyError(f"Train AnnData.obs need '{need}'")

# Preprocess like deconv: CP10k → joint winsor → asinh
ad_tr, ad_te = joint_cp10k_winsor_asinh(
    ad_tr, ad_te,
    do_winsor=GLOBAL_WINSORIZE, q=GLOBAL_WINSOR_Q,
    do_asinh=GLOBAL_ASINH, c=GLOBAL_ASINH_C
)

# Concat for joint HVG/Harmony
adata_all = ad_tr.concatenate(ad_te, join="inner", batch_key=BATCH_KEY_ALL, batch_categories=["train","test"])

# ---------- Markers FIRST (train-only; CP10k + log1p 僅用於選 marker) ----------
def _cp10k_log1p(ad: sc.AnnData) -> sc.AnnData:
    ad = ad.copy()
    sc.pp.normalize_total(ad, target_sum=1e4)
    return ad

# 先先對齊 gene（和特徵用的 ad_tr/ad_te 一樣的交集）
genes_shared = ad_tr.var_names.astype(str)  # 此時 ad_tr/ad_te 已被 joint_cp10k_winsor_asinh 切到共同基因
# 重新從原始檔讀一份 train（避免被 winsor/asinh 影響）
ad_tr_raw = sc.read_h5ad(TRAIN_AD_PATH)
# 限定到共同基因 + 只取 train
ad_tr_mark = ad_tr_raw[:, ad_tr_raw.var_names.astype(str).intersection(genes_shared)].copy()
# CP10k + log1p（僅用於 marker）
ad_tr_mark = _cp10k_log1p(ad_tr_mark)

# 用「train only + CP10k+log1p」算 pseudobulk→ComBat→log2FC→markers
pb = pseudobulk_by_sample_ct(ad_tr_mark, ct_col=CT_COL, sample_col=SAMPLE_COL)
pb = combat_on_pb(pb)
markers_by_ct = stratified_marker_by_ct(pb, topk_per_ct=MARKER_TOPK_CT, ct_order=TARGET_ORDER)

# 只保留存在於（之後要跑 Harmony 的）adata_all 基因集合中的 markers
# 注意：adata_all 是 winsor+asinh 後的 joint 物件，但 var_names 是同一組共同基因
markers_union = _safe_union_markers(markers_by_ct, adata_all.var_names)

# ---------- HVG on JOINT, then FORCE-IN markers ----------
sc.pp.highly_variable_genes(
    adata_all,
    n_top_genes=N_TOP_HVG,
    flavor="seurat_v3",
    batch_key=BATCH_KEY_ALL,
    inplace=True
)
hv_mask = adata_all.var["highly_variable"].values.copy()
name_to_pos = {g: i for i, g in enumerate(adata_all.var_names.astype(str))}
miss = [g for g in markers_union if g in name_to_pos and not hv_mask[name_to_pos[g]]]
if miss:
    for g in miss: hv_mask[name_to_pos[g]] = True  # expand HVG to include markers
genes_hvg_forced = adata_all.var_names[hv_mask].astype(str).tolist()

# ---------- Harmony on (HVG ∪ Markers) ----------
Z_hvg_mark = _harmony_on_subset(adata_all, genes_hvg_forced, BATCH_KEY_ALL, N_PCS_H)
hvg_cols = [f"harm_{i+1}" for i in range(Z_hvg_mark.shape[1])]

# ---------- Markers-only Harmony ----------
if len(markers_union) < 10:
    raise RuntimeError(f"Union markers too few: {len(markers_union)}")
Z_mark = _harmony_on_subset(adata_all, markers_union, BATCH_KEY_ALL, N_PCS_M)
mark_cols = [f"markharm_{i+1}" for i in range(Z_mark.shape[1])]

# ---------- Combined (z-score each block → concat) ----------
Z_hvg_use  = _stdz(Z_hvg_mark) if STANDARDIZE_BEFORE_COMBINE else Z_hvg_mark
Z_mark_use = _stdz(Z_mark)     if STANDARDIZE_BEFORE_COMBINE else Z_mark
Z_comb = np.concatenate([Z_hvg_use, Z_mark_use], axis=1)
comb_cols = hvg_cols + mark_cols

# ---------- Pack to DataFrame (with meta) ----------
if "pct_counts_mt" not in adata_all.obs.columns:
    adata_all.obs["pct_counts_mt"] = np.nan
meta = pd.DataFrame({
    "cell_type": adata_all.obs[LABEL_COL].astype(str).values,
    "Sample":    adata_all.obs[SAMPLE_COL].astype(str).values,
    "pct_counts_mt": pd.to_numeric(adata_all.obs["pct_counts_mt"], errors="coerce")
}, index=adata_all.obs_names)

df_hvg_forced = pd.DataFrame(Z_hvg_mark, index=adata_all.obs_names, columns=hvg_cols)
df_mark       = pd.DataFrame(Z_mark,       index=adata_all.obs_names, columns=mark_cols)
df_comb       = pd.DataFrame(Z_comb,       index=adata_all.obs_names, columns=comb_cols)

df_hvg_forced = pd.concat([meta, df_hvg_forced], axis=1)
df_mark       = pd.concat([meta, df_mark], axis=1)
df_comb       = pd.concat([meta, df_comb], axis=1)

hvg_path  = os.path.join(OUT_DIR, "features_hvgFORCE_markers_harmony.csv")
mark_path = os.path.join(OUT_DIR, "features_markers_harmony.csv")
comb_path = os.path.join(OUT_DIR, "features_combined_zscore.csv")

df_hvg_forced.to_csv(hvg_path)
df_mark.to_csv(mark_path)
df_comb.to_csv(comb_path)
print(f"[OK] wrote:\n - {hvg_path}  shape={df_hvg_forced.shape}\n - {mark_path}  shape={df_mark.shape}\n - {comb_path}  shape={df_comb.shape}")

# ---------- Quick sanity: show how many markers were forced-in ----------
n_hvg_before = N_TOP_HVG
n_forced = len(miss)
print(f"[INFO] HVG baseline={n_hvg_before}, forced-in markers not in HVG={n_forced}, final HVG∪Markers size={len(genes_hvg_forced)}")


2025-11-26 15:18:44,974 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2025-11-26 15:18:52,053 - harmonypy - INFO - sklearn.KMeans initialization complete.
2025-11-26 15:18:52,237 - harmonypy - INFO - Iteration 1 of 10
2025-11-26 15:19:14,648 - harmonypy - INFO - Iteration 2 of 10
2025-11-26 15:19:34,725 - harmonypy - INFO - Iteration 3 of 10
2025-11-26 15:19:48,797 - harmonypy - INFO - Iteration 4 of 10
2025-11-26 15:20:01,161 - harmonypy - INFO - Iteration 5 of 10
2025-11-26 15:20:12,065 - harmonypy - INFO - Converged after 5 iterations
2025-11-26 15:20:15,213 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2025-11-26 15:20:22,787 - harmonypy - INFO - sklearn.KMeans initialization complete.
2025-11-26 15:20:22,947 - harmonypy - INFO - Iteration 1 of 10
2025-11-26 15:20:34,149 - harmonypy - INFO - Iteration 2 of 10
2025-11-26 15:20:45,578 - harmonypy - INFO - Iteration 3 of 10
2025-11-26 15:20:57,001 - harmonypy - INFO - Iteration 4 of 

[OK] wrote:
 - ./outputs_features_csv/new_5/features_hvgFORCE_markers_harmony.csv  shape=(50990, 53)
 - ./outputs_features_csv/new_5/features_markers_harmony.csv  shape=(50990, 53)
 - ./outputs_features_csv/new_5/features_combined_zscore.csv  shape=(50990, 103)
[INFO] HVG baseline=2000, forced-in markers not in HVG=217, final HVG∪Markers size=2217


In [1]:
# file: scripts/preproc_clustering_like_deconv.py
# Make clustering features share the SAME preprocess as your deconvolution:
# CP10k -> joint per-gene winsorize -> asinh; markers-first, then HVG, and FORCE-IN markers to HVG.

import os, warnings
from typing import Dict, List
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# ============== Config ==============
TRAIN_AD_PATH = "../train_data/train_adata.h5ad"
TEST_AD_PATH  = "../test_data/test_adata.h5ad"

LABEL_COL     = "highLevelType"
SAMPLE_COL    = "Sample"
CT_COL        = "highLevelType"
BATCH_KEY_ALL = "set"              # 'train' / 'test'

TARGET_ORDER   = ["T","B","Endothelial","Fibroblast","Plasmablast","Myofibroblast","NK","Myeloid","Mast"]
MARKER_TOPK_CT = 25
N_TOP_HVG      = 2000              # baseline HVG size (may expand after forcing markers)
N_PCS_H          = 50
N_PCS_M          = 50

STANDARDIZE_BEFORE_COMBINE = True

# match your deconv preprocess
GLOBAL_WINSORIZE = True
GLOBAL_WINSOR_Q  = 0.995
GLOBAL_ASINH     = True
GLOBAL_ASINH_C   = 1

OUT_DIR = "./outputs_features_csv/new_7/"
os.makedirs(OUT_DIR, exist_ok=True)

# ============== Utils ==============
def _to_dense32(X):
    """
    Robust 2D dense float32 (C-order):
    - 支援 scipy.sparse (用 .A / .toarray())
    - NumPy 2.0：允許必要時複製，之後再強制 C-contiguous
    """
    # 1) 取 dense
    X = X.A if hasattr(X, "A") else (X.toarray() if hasattr(X, "toarray") else X)
    # 2) 允許必要時複製來轉 dtype（避免 np.array(..., copy=False) 的報錯）
    X = np.asarray(X, dtype=np.float32)
    # 3) 保證 C-order，避免之後串接/quantile 的奇怪坑
    if not X.flags.c_contiguous:
        X = np.ascontiguousarray(X)
    # 4) 形狀檢查
    if X.ndim != 2:
        raise ValueError(f"期望 2D 矩陣，拿到 shape={X.shape}")
    return X



def joint_cp10k_winsor_asinh(
    ad_tr: sc.AnnData,
    ad_te: sc.AnnData,
    do_winsor: bool = True,
    q: float = 0.995,
    do_asinh: bool = True,
    c: float = 1.0
) -> tuple[sc.AnnData, sc.AnnData]:
    """
    CP10k（不 log1p）→ 在 train+test 聯合空間對每個「基因」做 winsor（上分位裁切）→ 可選 asinh。
    保證：
      - train/test 用同一組且同順序的基因
      - X 一律為 dense np.ndarray（float32，C-order）
      - per-gene 的上分位向量 up 與 (n_genes,) 對齊；broadcast 安全
      - 若 numpy 路徑仍出錯，退回到 pandas 的 quantile 計算
    """
    # 1) 對齊 gene 交集與順序
    genes_tr = ad_tr.var_names.astype(str)
    genes_te = ad_te.var_names.astype(str)
    genes = genes_tr.intersection(genes_te)
    if len(genes) == 0:
        raise ValueError("Train/Test 沒有共同基因。")
    ad_tr = ad_tr[:, genes].copy()
    ad_te = ad_te[:, genes].copy()

    # 2) CP10k（不 log1p；後面會 asinh）
    sc.pp.normalize_total(ad_tr, target_sum=1e4)
    sc.pp.normalize_total(ad_te, target_sum=1e4)

    # 3) 強制 dense + float32 + C-order

    Xtr = _to_dense32(ad_tr.X)
    Xte = _to_dense32(ad_te.X)

    # 4) 形狀護欄
    if Xtr.shape[1] != Xte.shape[1]:
        raise ValueError(f"train/test 基因數不一致：{Xtr.shape[1]} vs {Xte.shape[1]}")
    n_genes = Xtr.shape[1]

    # 5) 聯合 winsor（per-gene）
    if do_winsor:
        if not (0.5 < q < 1.0):
            raise ValueError(f"winsor 分位數 q 應在 (0.5,1.0)，目前 q={q}")
        try:
            # 最常見且最快的路徑
            joint = np.concatenate([Xtr, Xte], axis=0)             # (n_cells_all × n_genes)，保證是 ndarray
            up = np.quantile(joint, q, axis=0).astype(np.float32)  # (n_genes,)
        except Exception as e:
            # 退避：改用 pandas 計算 per-gene 分位
            # 避免任何 __array_function__ 的坑（例如隱性 sparse）
            df_tr = pd.DataFrame(Xtr)
            df_te = pd.DataFrame(Xte)
            up = pd.concat([df_tr, df_te], axis=0).quantile(q, axis=0).astype(np.float32).values
        if up.shape != (n_genes,):
            raise ValueError(f"內部錯誤：up 形狀應為 ({n_genes},)，拿到 {up.shape}")

        # 明確 broadcast，避免 numpy 的隱式規則出意外
        np.minimum(Xtr, np.broadcast_to(up, Xtr.shape), out=Xtr)
        np.minimum(Xte, np.broadcast_to(up, Xte.shape), out=Xte)
        np.clip(Xtr, 0.0, None, out=Xtr)
        np.clip(Xte, 0.0, None, out=Xte)

    # 6) asinh 變換（可選）
    if do_asinh:
        invc = 1.0 / float(c)
        Xtr = np.arcsinh(Xtr * invc, dtype=np.float32)
        Xte = np.arcsinh(Xte * invc, dtype=np.float32)

    # 7) 回寫
    ad_tr.X = Xtr
    ad_te.X = Xte
    return ad_tr, ad_te


def pseudobulk_by_sample_ct(ad: sc.AnnData, ct_col: str, sample_col: str) -> pd.DataFrame:
    # 保證 meta 欄位是字串，避免 groupby 出怪
    obs = ad.obs[[ct_col, sample_col]].astype(str).copy()

    cols, names = [], []
    # 用 pandas 的 groupby(indices) 取得每個 (sample, ct) 的 cell 索引
    for (s, ct), idx in obs.groupby([sample_col, ct_col]).indices.items():
        if len(idx) == 0:
            continue
        # 取出子矩陣（保持 2D）
        Xsub = ad.X[idx, :]
        # to dense
        if hasattr(Xsub, "toarray"):
            Xsub = Xsub.toarray()
        elif hasattr(Xsub, "A"):
            Xsub = Xsub.A
        else:
            Xsub = np.asarray(Xsub)
        # 強制 float32、C-order、2D 形狀 (n_cells × n_genes)
        Xsub = np.asarray(Xsub, dtype=np.float32, order="C")
        if Xsub.ndim == 0:
            # 單細胞且單基因時可能成 scalar；改成 (1×1)
            Xsub = Xsub.reshape(1, 1)
        elif Xsub.ndim == 1:
            # 只有一個細胞或只剩一個基因 → 變成 (1 × n_genes)
            Xsub = Xsub.reshape(1, -1)

        # 平均成 pseudobulk（1 × n_genes），塞到 cols
        cols.append(Xsub.mean(axis=0).ravel())
        names.append(f"{s}|{ct}")

    if not cols:
        raise ValueError("pseudobulk_by_sample_ct 產生空結果：檢查 Sample 與 CT 欄位是否正確，或是否沒有交集細胞。")

    return pd.DataFrame(
        np.column_stack(cols),
        index=ad.var_names.astype(str),
        columns=names
    )

def combat_on_pb(pb: pd.DataFrame) -> pd.DataFrame:
    ad = sc.AnnData(pb.T.copy())
    ad.obs["batch"] = pd.Categorical([c.split("|",1)[0] for c in pb.columns])  # batch = Sample
    sc.pp.combat(ad, key="batch")
    return pd.DataFrame(ad.X.T, index=pb.index, columns=pb.columns)

def mean_ratio_markers(pb: pd.DataFrame, topk_per_ct: int, ct_order: list[str],
                       pseudocount: float = 1e-9, min_cols_per_ct: int = 1) -> list[str]:
    """
    Mean Ratio 選標法（每型 Top-K）：
      score_g,ct = ( mean_expr_g_in_ct + eps ) / ( max_{ct'!=ct} mean_expr_g_in_ct' + eps )
    參數
      pb: pseudobulk 矩陣（genes × columns），columns 形如 "Sample|CellType"
      topk_per_ct: 每個 cell type 挑 Top-K 基因
      ct_order: 想要的 cell type 順序（缺的會自動忽略，多的會追加在後）
      pseudocount: 避免 0 的微小常數
      min_cols_per_ct: 每個 cell type 至少需要的欄數（不足則跳過）
    回傳
      markers（list[str]）
    """
    # 解析出所有 cell types
    ct_all = sorted({c.split("|", 1)[1] for c in pb.columns})
    order = [ct for ct in ct_order if ct in ct_all] + [ct for ct in ct_all if ct not in ct_order]

    # 預先算：每個 cell type 的「基因×該型平均表達」
    mean_by_ct = {}
    for ct in ct_all:
        cols = [c for c in pb.columns if c.split("|", 1)[1] == ct]
        if len(cols) >= min_cols_per_ct:
            mean_by_ct[ct] = pb[cols].mean(axis=1)
    # 如果不足兩個 cell type 有平均值，就回傳空
    if len(mean_by_ct) < 2:
        return []

    sel = set()
    for ct in order:
        if ct not in mean_by_ct:
            continue
        m_ct = mean_by_ct[ct]

        # 其他 cell type 的平均表達 -> 逐基因取最大值
        others = [v for k, v in mean_by_ct.items() if k != ct]
        m_other_max = pd.concat(others, axis=1).max(axis=1)

        # Mean Ratio 分數（越大越專一）
        score = (m_ct + pseudocount) / (m_other_max + pseudocount)
        score = score.sort_values(ascending=False)

        sel.update(score.index[:topk_per_ct])

    return list(sel)


def _safe_union_markers(m_by_ct: Dict[str, List[str]], var_names) -> List[str]:
    s = set()
    for gl in m_by_ct.values(): s.update(gl)
    return sorted([g for g in s if g in set(var_names)])

def _stdz(X: np.ndarray) -> np.ndarray:
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True) + 1e-9
    return (X - mu) / sd

def _harmony_on_subset(adata: sc.AnnData, genes: List[str], batch_key: str, n_pcs: int) -> np.ndarray:
    ad = adata[:, genes].copy()
    sc.pp.scale(ad, max_value=10)
    sc.tl.pca(ad, n_comps=min(n_pcs, ad.n_vars, ad.n_obs), svd_solver="arpack")
    sc.external.pp.harmony_integrate(ad, key=batch_key)
    if "X_pca_harmony" not in ad.obsm:
        raise RuntimeError("Harmony on subset failed.")
    return ad.obsm["X_pca_harmony"]

# ============== Pipeline ==============
# Load
ad_tr = sc.read_h5ad(TRAIN_AD_PATH)
ad_te = sc.read_h5ad(TEST_AD_PATH)
for need in (LABEL_COL, SAMPLE_COL, CT_COL):
    if need not in ad_tr.obs: raise KeyError(f"Train AnnData.obs need '{need}'")

# Preprocess like deconv: CP10k → joint winsor → asinh
ad_tr, ad_te = joint_cp10k_winsor_asinh(
    ad_tr, ad_te,
    do_winsor=GLOBAL_WINSORIZE, q=GLOBAL_WINSOR_Q,
    do_asinh=GLOBAL_ASINH, c=GLOBAL_ASINH_C
)

# Concat for joint HVG/Harmony
adata_all = ad_tr.concatenate(ad_te, join="inner", batch_key=BATCH_KEY_ALL, batch_categories=["train","test"])

# ---------- Markers FIRST (train-only; CP10k + log1p 僅用於選 marker) ----------
def _cp10k_log1p(ad: sc.AnnData) -> sc.AnnData:
    ad = ad.copy()
    sc.pp.normalize_total(ad, target_sum=1e4)
    return ad

# 先先對齊 gene（和特徵用的 ad_tr/ad_te 一樣的交集）
genes_shared = ad_tr.var_names.astype(str)  # 此時 ad_tr/ad_te 已被 joint_cp10k_winsor_asinh 切到共同基因
# 重新從原始檔讀一份 train（避免被 winsor/asinh 影響）
ad_tr_raw = sc.read_h5ad(TRAIN_AD_PATH)
# 限定到共同基因 + 只取 train
ad_tr_mark = ad_tr_raw[:, ad_tr_raw.var_names.astype(str).intersection(genes_shared)].copy()
# CP10k + log1p（僅用於 marker）
ad_tr_mark = _cp10k_log1p(ad_tr_mark)

# 用「train only + CP10k+log1p」算 pseudobulk→ComBat→log2FC→markers
pb = pseudobulk_by_sample_ct(ad_tr_mark, ct_col=CT_COL, sample_col=SAMPLE_COL)
pb = combat_on_pb(pb)
# markers_by_ct = stratified_marker_by_ct(pb, topk_per_ct=MARKER_TOPK_CT, ct_order=TARGET_ORDER)
#         # 如果你已改成 Mean Ratio 選標（上一訊息給的函式）
# 仍然得到一個 list
markers_list = mean_ratio_markers(pb, topk_per_ct=MARKER_TOPK_CT, ct_order=TARGET_ORDER,
                                  pseudocount=1e-9, min_cols_per_ct=1)
# 去重、保序
markers_list = list(dict.fromkeys(markers_list))

# 直接和 adata_all.var_names 取交集（不需要 _safe_union_markers）
allowed = set(map(str, adata_all.var_names))
markers_union = [g for g in markers_list if g in allowed]


# ---------- HVG on JOINT, then FORCE-IN markers ----------
sc.pp.highly_variable_genes(
    adata_all,
    n_top_genes=N_TOP_HVG,
    flavor="seurat_v3",
    batch_key=BATCH_KEY_ALL,
    inplace=True
)
hv_mask = adata_all.var["highly_variable"].values.copy()
name_to_pos = {g: i for i, g in enumerate(adata_all.var_names.astype(str))}
miss = [g for g in markers_union if g in name_to_pos and not hv_mask[name_to_pos[g]]]
if miss:
    for g in miss: hv_mask[name_to_pos[g]] = True  # expand HVG to include markers
genes_hvg_forced = adata_all.var_names[hv_mask].astype(str).tolist()

# ---------- Harmony on (HVG ∪ Markers) ----------
Z_hvg_mark = _harmony_on_subset(adata_all, genes_hvg_forced, BATCH_KEY_ALL, N_PCS_H)
hvg_cols = [f"harm_{i+1}" for i in range(Z_hvg_mark.shape[1])]

# ---------- Markers-only Harmony ----------
if len(markers_union) < 10:
    raise RuntimeError(f"Union markers too few: {len(markers_union)}")
Z_mark = _harmony_on_subset(adata_all, markers_union, BATCH_KEY_ALL, N_PCS_M)
mark_cols = [f"markharm_{i+1}" for i in range(Z_mark.shape[1])]

# ---------- Combined (z-score each block → concat) ----------
Z_hvg_use  = _stdz(Z_hvg_mark) if STANDARDIZE_BEFORE_COMBINE else Z_hvg_mark
Z_mark_use = _stdz(Z_mark)     if STANDARDIZE_BEFORE_COMBINE else Z_mark
Z_comb = np.concatenate([Z_hvg_use, Z_mark_use], axis=1)
comb_cols = hvg_cols + mark_cols

# ---------- Pack to DataFrame (with meta) ----------
if "pct_counts_mt" not in adata_all.obs.columns:
    adata_all.obs["pct_counts_mt"] = np.nan
meta = pd.DataFrame({
    "cell_type": adata_all.obs[LABEL_COL].astype(str).values,
    "Sample":    adata_all.obs[SAMPLE_COL].astype(str).values,
    "pct_counts_mt": pd.to_numeric(adata_all.obs["pct_counts_mt"], errors="coerce")
}, index=adata_all.obs_names)

df_hvg_forced = pd.DataFrame(Z_hvg_mark, index=adata_all.obs_names, columns=hvg_cols)
df_mark       = pd.DataFrame(Z_mark,       index=adata_all.obs_names, columns=mark_cols)
df_comb       = pd.DataFrame(Z_comb,       index=adata_all.obs_names, columns=comb_cols)

df_hvg_forced = pd.concat([meta, df_hvg_forced], axis=1)
df_mark       = pd.concat([meta, df_mark], axis=1)
df_comb       = pd.concat([meta, df_comb], axis=1)

hvg_path  = os.path.join(OUT_DIR, "features_hvgFORCE_markers_harmony.csv")
mark_path = os.path.join(OUT_DIR, "features_markers_harmony.csv")
comb_path = os.path.join(OUT_DIR, "features_combined_zscore.csv")

df_hvg_forced.to_csv(hvg_path)
df_mark.to_csv(mark_path)
df_comb.to_csv(comb_path)
print(f"[OK] wrote:\n - {hvg_path}  shape={df_hvg_forced.shape}\n - {mark_path}  shape={df_mark.shape}\n - {comb_path}  shape={df_comb.shape}")

# ---------- Quick sanity: show how many markers were forced-in ----------
n_hvg_before = N_TOP_HVG
n_forced = len(miss)
print(f"[INFO] HVG baseline={n_hvg_before}, forced-in markers not in HVG={n_forced}, final HVG∪Markers size={len(genes_hvg_forced)}")


2025-11-26 16:02:41,790 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2025-11-26 16:02:55,961 - harmonypy - INFO - sklearn.KMeans initialization complete.
2025-11-26 16:02:56,179 - harmonypy - INFO - Iteration 1 of 10
2025-11-26 16:04:53,782 - harmonypy - INFO - Iteration 2 of 10
2025-11-26 16:06:18,540 - harmonypy - INFO - Iteration 3 of 10
2025-11-26 16:06:46,895 - harmonypy - INFO - Iteration 4 of 10
2025-11-26 16:07:03,581 - harmonypy - INFO - Iteration 5 of 10
2025-11-26 16:07:16,622 - harmonypy - INFO - Converged after 5 iterations
2025-11-26 16:07:18,199 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2025-11-26 16:07:25,744 - harmonypy - INFO - sklearn.KMeans initialization complete.
2025-11-26 16:07:25,906 - harmonypy - INFO - Iteration 1 of 10
2025-11-26 16:07:39,812 - harmonypy - INFO - Iteration 2 of 10
2025-11-26 16:07:53,352 - harmonypy - INFO - Iteration 3 of 10
2025-11-26 16:09:02,893 - harmonypy - INFO - Iteration 4 of 

[OK] wrote:
 - ./outputs_features_csv/new_7/features_hvgFORCE_markers_harmony.csv  shape=(50990, 53)
 - ./outputs_features_csv/new_7/features_markers_harmony.csv  shape=(50990, 53)
 - ./outputs_features_csv/new_7/features_combined_zscore.csv  shape=(50990, 103)
[INFO] HVG baseline=2000, forced-in markers not in HVG=25, final HVG∪Markers size=2025


In [3]:
markers_by_ct

['PLA2G2A',
 'IGHA1',
 'LYZ',
 'BANK1',
 'MS4A1',
 'IGHG2',
 'KLRD1',
 'KIT',
 'KLRC2',
 'CD5',
 'CLEC10A',
 'AC108134.2',
 'FCN1',
 'SDR42E2',
 'VWF',
 'IER5L',
 'F3',
 'RTKN2',
 'PTGDS',
 'PDGFRA',
 'IGLV2-14',
 'XCL1',
 'CLDN5',
 'TNFRSF4',
 'ACKR1',
 'NEU1',
 'TPSAB1',
 'FOXP3',
 'SLC18A2',
 'COL12A1',
 'XCL2',
 'IL1B',
 'MMRN1',
 'ADGRF5',
 'LAIR2',
 'AC245128.3',
 'ZC3H12D',
 'IGLV2-23',
 'GALC',
 'SOX18',
 'KRT86',
 'PCAT19',
 'COL6A3',
 'LINC01781',
 'CXCL14',
 'TNF',
 'CD300E',
 'HSPA6',
 'IGLC2',
 'HSPA1B',
 'CSF2RA',
 'CCL11',
 'CTSG',
 'CSKMT',
 'BLK',
 'NKG7',
 'RARRES2',
 'BATF',
 'GALNT6',
 'ECSCR',
 'JCHAIN',
 'EMCN',
 'PRF1',
 'CCL21',
 'LINC00926',
 'HDC',
 'VPREB3',
 'NCF2',
 'IGLV3-19',
 'LINC01857',
 'TRAT1',
 'DNAJB1',
 'CSF1R',
 'CPA3',
 'GATA2',
 'TRDC',
 'AIF1',
 'IGLV3-21',
 'GNLY',
 'LINC01943',
 'CRYAB',
 'LY9',
 'IGHG3',
 'KLRC1',
 'ARHGAP24',
 'IGLL5',
 'TNFRSF13C',
 'IL1RL1',
 'AGT',
 'CTLA4']

In [2]:
# file: scripts/tune_multiclass_optuna_lgbm_safe.py
# -*- coding: utf-8 -*-
"""
SAFE Optuna tuning for multiclass LightGBM under GroupKFold (no binary stage).
- 只調 multiclass 參數（移除 binary mask）
- GroupKFold，若某 fold 訓練集單一類別 → 跳過該 fold（不報錯）
- 目標分數：0.5*(ARI + V)
- 完全靜音 LightGBM/Optuna 訓練輸出，僅列印 trial 結果一行
- 匯出最佳參數 JSON
"""

import os, json, warnings, io, contextlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from lightgbm.callback import early_stopping, log_evaluation
from lightgbm.basic import LightGBMError
from typing import Dict, Optional

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import adjusted_rand_score, v_measure_score

# ======== Paths / Columns ========
IN_DIR   = "./outputs_features_csv/new_5/"
OUT_DIR  = "./outputs_features_csv/new_5/"
FILE     = "features_combined_zscore.csv"

LABEL_COL = "cell_type"
GROUP_COL = "Sample"

# ======== CV / Seed / Tuning budget ========
N_SPLITS = 5
SEED     = 42
EARLY_STOP_ROUNDS = 50
N_TRIALS = 120
TIMEOUT  = None

# ======== Logging ========
optuna.logging.set_verbosity(optuna.logging.ERROR)

@contextlib.contextmanager
def _silent_io():
    buf_out, buf_err = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(buf_out), contextlib.redirect_stderr(buf_err):
        yield

# ======== Utils ========
def load_full(path: str) -> pd.DataFrame:
    if not os.path.exists(path): raise FileNotFoundError(path)
    df = pd.read_csv(path, index_col=0)
    if df.empty: raise ValueError(f"Empty: {path}")
    return df

def split_masks(df: pd.DataFrame, label_col: str):
    if label_col not in df.columns:
        return pd.Series(False, index=df.index), pd.Series(True, index=df.index)
    lbl = df[label_col]
    is_lab = (lbl.notna() & lbl.astype(str).str.strip().ne("") & lbl.astype(str).str.lower().ne("nan"))
    return is_lab, ~is_lab

def numeric_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    Xdf = df.select_dtypes(include=[np.number]).copy()
    Xdf = Xdf.loc[:, Xdf.nunique(dropna=True) > 1]
    if Xdf.shape[1] == 0: raise ValueError("No usable numeric features.")
    return Xdf.fillna(0.0)

def score_ari_v(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return 0.5 * (adjusted_rand_score(y_true, y_pred) + v_measure_score(y_true, y_pred))

def _has_two_classes(y: np.ndarray) -> bool:
    return np.unique(y).size >= 2

# ======== search space（安全可攜） ========
def make_params_mul(trial: optuna.Trial) -> Dict:
    bt = trial.suggest_categorical("boosting_type", ["gbdt", "goss", "dart"])
    params = dict(
        objective="multiclass",
        boosting_type=bt,
        # 讓 early_stopping 能工作；也可以把這個參數納入搜尋
        n_estimators=300,  # <-- 固定 500
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        num_leaves=trial.suggest_int("num_leaves", 1, 255),
        max_depth=trial.suggest_categorical("max_depth", [-1, 4, 6, 8, 10, 12, 14]),
        min_child_samples=trial.suggest_int("min_child_samples", 1, 120),
        min_child_weight=trial.suggest_float("min_child_weight", 1e-3, 100.0, log=True),
        min_split_gain=trial.suggest_float("min_split_gain", 0.0, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        max_bin=trial.suggest_int("max_bin", 63, 511),
        feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),
        verbosity=-1, random_state=SEED, n_jobs=-1
    )
    # bagging: only for gbdt/dart
    if bt in ("gbdt", "dart"):
        params["bagging_fraction"] = trial.suggest_float("bagging_fraction", 0.5, 1.0)
        params["bagging_freq"]     = trial.suggest_int("bagging_freq", 0, 10)
    else:
        params["bagging_fraction"] = 1.0
        params["bagging_freq"]     = 0
    # dart extra
    if bt == "dart":
        params["drop_rate"] = trial.suggest_float("drop_rate", 0.01, 0.5)
        params["skip_drop"] = trial.suggest_float("skip_drop", 0.0, 0.5)
        params["max_drop"]  = trial.suggest_int("max_drop", 20, 200)
    return params

# ======== objective ========
def objective_factory(X: np.ndarray, y_str: np.ndarray, groups: np.ndarray, le_global: LabelEncoder):
    gkf = GroupKFold(n_splits=N_SPLITS)
    K_global = len(le_global.classes_)

    def objective(trial: optuna.Trial) -> float:
        try:
            params_mul = make_params_mul(trial)

            y_true_oof = np.full(y_str.shape[0], -1, dtype=int)
            y_pred_oof = np.full(y_str.shape[0], -1, dtype=int)
            valid_folds = 0

            for tr_idx, va_idx in gkf.split(X, y_str, groups):
                # fold 內局部編碼（避免某些類別在此 fold 缺失）
                le_fold = LabelEncoder().fit(np.unique(np.r_[y_str[tr_idx], y_str[va_idx]]))
                y_tr_fold = le_fold.transform(y_str[tr_idx])
                y_va_fold = le_fold.transform(y_str[va_idx])

                # 訓練集至少要有兩個類別，否則跳過
                if not _has_two_classes(y_tr_fold):
                    continue

                fold_to_global = le_global.transform(le_fold.classes_)
                pm = dict(params_mul)
                pm["num_class"] = len(le_fold.classes_)

                clf_mul = lgb.LGBMClassifier(**pm)
                with _silent_io():
                    clf_mul.fit(
                        X[tr_idx], y_tr_fold,
                        eval_set=[(X[va_idx], y_va_fold)],
                        eval_metric="multi_logloss",
                        callbacks=[log_evaluation(period=0), early_stopping(EARLY_STOP_ROUNDS, verbose=False)]
                    )
                it_mul = clf_mul.best_iteration_ or pm.get("n_estimators", 100)
                proba = clf_mul.predict_proba(X[va_idx], num_iteration=it_mul)
                yhat_fold = np.argmax(proba, axis=1)
                yhat_global = fold_to_global[yhat_fold]

                y_true_oof[va_idx] = le_global.transform(y_str[va_idx])
                y_pred_oof[va_idx] = yhat_global
                valid_folds += 1

            if valid_folds == 0:
                raise optuna.TrialPruned()

            mask = (y_true_oof >= 0) & (y_pred_oof >= 0)
            return score_ari_v(y_true_oof[mask], y_pred_oof[mask])

        except LightGBMError as e:
            trial.set_user_attr("lgbm_error", str(e))
            raise optuna.TrialPruned()
        except Exception as e:
            trial.set_user_attr("error", repr(e))
            raise optuna.TrialPruned()

    return objective

# ======== callback: concise per-trial line ========
def _short_params(d: dict, ndigits: int = 4) -> dict:
    out = {}
    for k, v in d.items():
        if isinstance(v, float):
            out[k] = round(v, ndigits)
        else:
            out[k] = v
    return out

def on_trial_end(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    if trial.state == optuna.trial.TrialState.COMPLETE:
        print(f"[trial {trial.number}] value={trial.value:.6f}  params={_short_params(trial.params)}")
    elif trial.state == optuna.trial.TrialState.PRUNED:
        print(f"[trial {trial.number}] status=PRUNED")
    else:
        print(f"[trial {trial.number}] status={trial.state.name}")


# ======== main ========
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    df = load_full(os.path.join(IN_DIR, FILE))
    is_lab, _ = split_masks(df, LABEL_COL)
    if GROUP_COL not in df.columns: raise KeyError(f"Missing '{GROUP_COL}'")
    if not is_lab.any(): raise ValueError("No labeled rows.")

    X_all = numeric_feature_matrix(df).to_numpy(dtype=float)
    y_all = df[LABEL_COL].astype(str).values
    idx_all = df.index.to_numpy()

    X_lab, y_lab = X_all[is_lab.values], y_all[is_lab.values]
    groups = df.loc[idx_all[is_lab.values], GROUP_COL].astype(str).values

    le = LabelEncoder().fit(y_lab)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(
        objective_factory(X_lab, y_lab, groups, le),
        n_trials=N_TRIALS, timeout=TIMEOUT, n_jobs=1,
        callbacks=[on_trial_end]
    )

    best = study.best_trial
    P = best.params

    best_mul = dict(
        objective="multiclass",
        boosting_type=P["boosting_type"],
        n_estimators=300,  # <-- 固定 500
        learning_rate=P["learning_rate"],
        num_leaves=P["num_leaves"],
        max_depth=P["max_depth"],
        min_child_samples=P["min_child_samples"],
        min_child_weight=P["min_child_weight"],
        min_split_gain=P["min_split_gain"],
        reg_alpha=P["reg_alpha"],
        reg_lambda=P["reg_lambda"],
        max_bin=P["max_bin"],
        feature_fraction=P["feature_fraction"],
        bagging_fraction=(P.get("bagging_fraction", 1.0)),
        bagging_freq=(P.get("bagging_freq", 0)),
        verbosity=-1, random_state=SEED, n_jobs=-1
    )
    if P["boosting_type"] == "dart":
        best_mul.update(dict(drop_rate=P["drop_rate"], skip_drop=P["skip_drop"], max_drop=P["max_drop"]))

    out = dict(
        best_score=float(best.value),
        best_multi=best_mul,
        seed=SEED, n_splits=N_SPLITS, file=FILE, label_col=LABEL_COL, group_col=GROUP_COL
    )
    path = os.path.join(OUT_DIR, "tuned_lgbm_multiclass_optuna.json")
    with open(path, "w") as f: json.dump(out, f, indent=2)

    print(f"[BEST] value={best.value:.6f}")
    print(f"[OK] wrote: {path}")

if __name__ == "__main__":
    main()


[trial 0] value=0.840883  params={'boosting_type': 'goss', 'learning_rate': 0.0455, 'num_leaves': 40, 'max_depth': 14, 'min_child_samples': 100, 'min_child_weight': 0.0115, 'min_split_gain': 0.1818, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'max_bin': 298, 'feature_fraction': 0.716}
[trial 1] value=0.839929  params={'boosting_type': 'goss', 'learning_rate': 0.0147, 'num_leaves': 94, 'max_depth': 4, 'min_child_samples': 21, 'min_child_weight': 0.0021, 'min_split_gain': 0.9489, 'reg_alpha': 4.9056, 'reg_lambda': 0.1886, 'max_bin': 199, 'feature_fraction': 0.5488}
[trial 2] value=0.836073  params={'boosting_type': 'gbdt', 'learning_rate': 0.0311, 'num_leaves': 9, 'max_depth': -1, 'min_child_samples': 117, 'min_child_weight': 7.5104, 'min_split_gain': 0.9395, 'reg_alpha': 1.131, 'reg_lambda': 0.0024, 'max_bin': 476, 'feature_fraction': 0.5442, 'bagging_fraction': 0.598, 'bagging_freq': 0}
[trial 3] value=0.834787  params={'boosting_type': 'goss', 'learning_rate': 0.1063, 'num_leaves': 91, 'max_